In [ ]:
import torch
import torch.nn as nn
from torch.nn import init


class ChannelAttention(nn.Module):
    """
    Channel attention branch.

    Input:
        x: Tensor of shape (B, C, H, W)

    Output:
        channel_weight: Tensor of shape (B, C, 1, 1)

    Notes:
        - Global average pooling and global max pooling are both used to describe
          channel-wise responses.
        - The two pooled descriptors share the same MLP.
        - The output is a channel weight map in [0, 1].
    """
    def __init__(self, in_channels: int, reduction: int = 2):
        super(ChannelAttention, self).__init__()

        if in_channels % reduction != 0:
            raise ValueError("in_channels must be divisible by reduction")

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.size()

        # Squeeze spatial dimensions first.
        avg_descriptor = self.avg_pool(x).view(b, c)
        max_descriptor = self.max_pool(x).view(b, c)

        # The same MLP is applied to both descriptors.
        avg_out = self.mlp(avg_descriptor)
        max_out = self.mlp(max_descriptor)

        # Fuse the two responses and restore them to (B, C, 1, 1).
        channel_weight = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return channel_weight


class SpatialAttention(nn.Module):
    """
    Spatial attention branch.

    Input:
        x: Tensor of shape (B, C, H, W)

    Output:
        spatial_weight: Tensor of shape (B, 1, H, W)

    Notes:
        - Channel-wise average pooling and max pooling are used to keep
          complementary spatial cues.
        - The pooled maps are concatenated and passed through a 7x7 convolution.
        - The output is a spatial weight map in [0, 1].
    """
    def __init__(self, kernel_size: int = 7):
        super(SpatialAttention, self).__init__()

        if kernel_size != 7:
            raise ValueError("kernel_size must be 7")

        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pool along the channel dimension.
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map, _ = torch.max(x, dim=1, keepdim=True)

        # Shape after concat: (B, 2, H, W)
        pooled = torch.cat([max_map, avg_map], dim=1)

        spatial_weight = self.sigmoid(self.conv(pooled))
        return spatial_weight


class PositionalAttention(nn.Module):
    """
    Positional attention branch.

    Input:
        x: Tensor of shape (B, C, H, W)

    Output:
        out: Tensor of shape (B, C, H, W)

    Notes:
        - Three 1x1 convolutions are used to generate three feature embeddings.
        - Two embeddings are used to build a position-to-position attention matrix.
        - The attention map is then used to reweight the third embedding.
        - A residual connection is applied at the end.
    """
    def __init__(self, in_channels: int):
        super(PositionalAttention, self).__init__()

        self.conv_c = nn.Conv2d(in_channels, in_channels, kernel_size=1, bias=False)
        self.conv_d = nn.Conv2d(in_channels, in_channels, kernel_size=1, bias=False)
        self.conv_e = nn.Conv2d(in_channels, in_channels, kernel_size=1, bias=False)

        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.size()
        n = h * w

        # Branch C: (B, C, H, W) -> (B, N, C)
        feat_c = self.conv_c(x).view(b, c, n).permute(0, 2, 1)

        # Branch D: (B, C, H, W) -> (B, C, N)
        feat_d = self.conv_d(x).view(b, c, n)

        # Branch E: (B, C, H, W) -> (B, N, C)
        feat_e = self.conv_e(x).view(b, c, n).permute(0, 2, 1)

        # Build attention between all spatial positions.
        # attention shape: (B, N, N)
        attention = torch.bmm(feat_c, feat_d)
        attention = self.softmax(attention)

        # Reweight the E branch with the learned attention.
        # out shape before reshape: (B, N, C)
        out = torch.bmm(attention, feat_e)

        # Restore the feature map layout.
        out = out.permute(0, 2, 1).contiguous().view(b, c, h, w)

        # Residual connection.
        out = out + x
        return out


class CSPAM(nn.Module):
    """
    Channel-Spatial-Positional Attention Module.

    Processing order:
        1) Channel attention
        2) Spatial attention
        3) Positional attention

    Input:
        x: Tensor of shape (B, C, H, W)

    Output:
        out: Tensor of shape (B, C, H, W)
    """
    def __init__(self, in_channels: int, reduction: int = 2):
        super(CSPAM, self).__init__()

        self.channel_attention = ChannelAttention(
            in_channels=in_channels,
            reduction=reduction
        )
        self.spatial_attention = SpatialAttention(kernel_size=7)
        self.positional_attention = PositionalAttention(in_channels=in_channels)

    def init_weights(self) -> None:
        """
        Initialize module parameters.

        Convolution:
            Kaiming initialization

        Linear:
            Small normal initialization

        BatchNorm:
            Weight = 1, Bias = 0
        """
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)

            elif isinstance(m, nn.Linear):
                init.normal_(m.weight, std=0.001)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

            elif isinstance(m, nn.BatchNorm2d):
                init.constant_(m.weight, 1)
                init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Channel attention
        channel_weight = self.channel_attention(x)
        x_channel = x * channel_weight

        # Spatial attention
        spatial_weight = self.spatial_attention(x_channel)
        x_spatial = x_channel * spatial_weight

        # Positional attention
        out = self.positional_attention(x_spatial)

        return out


if __name__ == "__main__":
    # Example
    x = torch.randn(1, 512, 28, 28)
    model = CSPAM(in_channels=512, reduction=2)
    model.init_weights()

    out = model(x)

    print("Input shape :", x.shape)
    print("Output shape:", out.shape)